In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@
#import useful libraries
#@@@@@@@@@@@@@@@@@@@@@@@
import numpy as np

from tqdm import tqdm
from IPython.display import clear_output
import time

import copy


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#two classes for these two environments
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
class Battleship:
    '''A class to manage the board'''
    
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def __init__(self, config):
        '''Set up the constructor
            Takes -- config, a dictionary specifying the track dimensions and initial state
        '''
        self.config = config
        self.true_state = np.zeros([config['n'],config['n']],dtype = int)    #the true positions of the ships
        self.board_state = np.zeros([config['n'],config['n']],dtype = int)    #the actual record of hits and misses
        if config['boards']:
            self.consistent_boards = config['boards']
            self.state = self.compute_posterior()#np.zeros([config['n'],config['n']],dtype = int) + 1    #TODO -- consider how to change this

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def position_maker(self, ship):
        '''A function to choose a position for a ship
            Takes:
                self -- class instance info above
                ship -- int, the index in the list of ship lengths
            Returns:
                an int, int pair, the coordinates of the first cell the ship occupies
        '''
        coord_1 = np.random.choice(list(range(self.config['n'] - self.config['ships'][ship] + 1)))    #choose first coordinate, guaranteed to have ship on board
        coord_2 = np.random.choice(list(range(self.config['n'])))    #choose second coordinate
        return coord_1,coord_2

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def check_empty(self, coord_1, coord_2, vert, ship):
        '''A function to make sure the proposed location is currently empty
            Takes:
                self -- class instance info above
                coord_1 -- int, the first coordinate, constrained to ensure the ship lies on the board
                coord_2 -- int, the second coordinate
                vert -- boolean, whether to orient the ship vertically or horizontally
                ship -- int, the index in the list of ship lengths
            Returns:
                a boolean, indicating if the placement is legal
        '''
        if vert:    #if the ship is oriented vertically...
            if sum(self.true_state[coord_1 + list(range(self.config['ships'][ship])),coord_2]) > 0:    #and if there is another ship lying across its path...
                empty = False    #set the empty indicator to false
            else:    #otherwise if the proposed location is empty...
                empty = True    #set the empty indicator to true
        else:    #if the ship is oriented horizontally...
            if sum(self.true_state[coord_2,coord_1 + list(range(self.config['ships'][ship]))]) > 0:    #and if there is anther ship lying across its path...
                empty = False    #set the empty indicator to false
            else:    #otherwise if the proposed location is empty...
                empty = True    #set the empty indicator to true
        return empty

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def place_ship(self, ship):
        '''A function to put a single ship on the board
            Takes:
                self -- class instance info above
                ship -- int, the index in the list of ship lengths
        '''
        empty = False    #init the stopping criterion
        while not empty:    #while we should keep going...
            coord_1,coord_2 = self.position_maker(ship)    #sample a position for the ship
            vert = np.random.uniform() < 0.5    #choose whether ship is horizontal or vertical
            empty = self.check_empty(coord_1,coord_2,vert,ship)    #check if legal
        if vert:    #if the ship is vertical...
            self.true_state[coord_1 + list(range(self.config['ships'][ship])),coord_2] = 1    #place on board
        else:    #if the ship is horizontal...
            self.true_state[coord_2,coord_1 + list(range(self.config['ships'][ship]))] = 1    #place on board
        vert = np.random.uniform() < 0.5    #choose whether ship is horizontal or vertical

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def make_board(self):
        '''A function to put all ships on the board
            Takes:
                self -- class instance info above
        '''
        for j in range(len(self.config['ships'])):    #loop over ship lengths
            self.place_ship(j)    #put on board'

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def reset(self):
        '''A function to reset the board
            Takes:
                self -- class instance info above
        '''
        self.true_state = np.zeros([self.config['n'],self.config['n']],dtype = int)    #reset the hidden state
        self.board_state = np.zeros([self.config['n'],self.config['n']],dtype = int)    #reset the observed state
        self.state = np.zeros([self.config['n'],self.config['n']],dtype = int) + 1    #TODO -- consider how to change this
        if self.config['boards']:
            self.consistent_boards = self.config['boards']
            self.state = self.compute_posterior()

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def check_win(self):
        '''A function to check whether the agent has won
            Takes:
                self -- class instance info above
            Returns:
                a boolean, indicating if the agent sank all the ships
        '''
        if np.sum(self.board_state>0) == sum(self.config['ships']):    #if the number of hits is equal to the total length of ships...
            return True    #return true -- the agent won!
        else:    #otherwise...
            return False    #return false -- the game is still going

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def reward(self, s_pr, s):
        #return np.sum((self.board_state > 0)*1) - np.sum((old_state > 0)*1)
        return round(np.log(np.sum(s)+1)) - round(np.log(np.sum(s_pr)+1))

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def compute_posterior(self):
        new_consistent = {}
        board_mask = (self.board_state == 0)*1
        posterior_distribution = np.zeros([self.config['n'],self.config['n']])    #set up a grid to track cell counts from consistent boards
        for k in self.consistent_boards.keys():    #loop over all currently consistent boards
            c1 = np.sum((self.consistent_boards[k] - self.board_state) < 0)    #check if the candidate board is empty where the obs board has a hit
            c2 = np.sum((self.board_state - self.consistent_boards[k]) < -1)    #check if the candidate board is filled where the obs board has a miss
            if c1 + c2 == 0:    #if neither condition holds...
                posterior_distribution = posterior_distribution + self.consistent_boards[k]    #add the information to the posterior
                new_consistent[k] = self.consistent_boards[k]
        self.consistent_boards = new_consistent
        posterior_distribution = posterior_distribution * board_mask    #zero out locations that have been tried already
        self.state = posterior_distribution
        return posterior_distribution
        
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def step(self, a):
        '''A function to update the state given the true enemy fleet locations'''
        #old_state = copy.deepcopy(self.board_state)
        old_state = copy.deepcopy(self.state)
        if self.true_state[a['row'],a['col']] > 0:    #if there is a ship at the chosen location...
            self.board_state[a['row'],a['col']] = 1    #record a hit on the observed state as a 1
        else:    #if there is no ship
            self.board_state[a['row'],a['col']] = -1    #record a miss on the observed state as a -1
        done = self.check_win()    #check to see if the agent has won
        posterior = self.compute_posterior()
        r = self.reward(s_pr = self.state,s = old_state)
        if np.sum(self.board_state == 1) == sum(self.config['ships']):
            r = 10
        return{'state':self.state
               ,'reward': r #self.reward(old_state)
               ,'done':done}
        

In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@
#a class for the environment
#@@@@@@@@@@@@@@@@@@@@@@@@@@@
class agent:
    '''A class to manage the agent'''
    
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def __init__(self, config):
        '''Set up the constructor
            Takes -- config, a dictionary specifying the track dimensions and initial state
        '''
        self.config = config    #save the config
        self.Q = {}    #init the dictionary of Q-values
        self.coordinates = []    #init the coordinates
        for i in range(config['n']):    #loop over rows
            for j in range(config['n']):    #loop over columns
                self.coordinates.append((i,j))    #add the tuple to the list of coordinates
        self.A = list(range(config['top_k']))    #create actions -- a ranking threshold
                
#     #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#     def update_Q(self,state,a):
#         if state in self.Q:
#             self.Q[state][a] += 1
#         else:
#             self.Q[state] = {k:np.random.uniform() for k in self.A}    

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def pi(self, state, epsilon):
        '''The policy to generate actions
            Takes:
                state -- the agent representation of the state, the logged number of possible boards
                epsilon -- float, the prob of a random action
        '''
        if np.random.uniform() < epsilon:    #with small probability...
            return self.A[np.random.choice(a = range(len(self.A)))]    #return a random action
        else:
            if state in self.Q:   #if this state has Q-values...
                return list(self.Q[state].keys())[np.argmax(list(self.Q[state].values()))]    #return a threshold according to the Q-values
            else:    #if this state does not yet have Q-values...
                return self.A[np.random.choice(a = range(len(self.A)))]    #return a random action
        
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def shoot_where(self, state, a):    
        state_flattened = state.flatten()    #flatten into a 1d array
        threshold = sorted(state_flattened,reverse = True)[a]    #sort and choose the rank
        state_flattened[(state_flattened < threshold)] = 0    #zero out everything under the rank
        state_flattened = state_flattened/sum(state_flattened)    #normalize
        return self.coordinates[np.random.choice(a = range(len(self.coordinates)),p = state_flattened)]    #return a random action according to the board state

#     #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#     def update_Q_SARSA(self,s,a,r_pr,s_pr,a_pr):
#         self.Q[str(s)][str(a)] = (self.Q[str(s)][str(a)] 
#                                     + self.config['alpha']
#                                       *(r_pr + self.config['gamma']*self.Q[str(s_pr)][str(a_pr)] - self.Q[str(s)][str(a)]))    #perform the SARSA update

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def update_Q_learning(self, s, a, r_s_pr_a_s, s_pr):
        if s_pr not in self.Q:
            self.Q[s_pr] = {k:np.random.uniform() for k in self.A}
        if s not in self.Q:
            self.Q[s] = {k:np.random.uniform() for k in self.A}
        else:
            self.Q[s][a] = (self.Q[s][a] + self.config['alpha']
                                      *(r_s_pr_a_s + self.config['gamma']*max(list(self.Q[s_pr].values())) - self.Q[s][a]))    #perform the Q-learning update

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def update_Expected_SARSA(self, s, a, r_s_pr_a_s, s_pr):
        if s_pr not in self.Q:
            self.Q[s_pr] = {k:np.random.uniform() for k in self.A}
        if s not in self.Q:
            self.Q[s] = {k:np.random.uniform() for k in self.A}
        else:
            pi = [self.config['epsilon']/2]*(len(self.Q[s_pr].values()))
            pi[np.argmax(list(self.Q[s_pr].values()))] = 1 - self.config['epsilon'] + self.config['epsilon']/2
            expectation = sum(np.array(pi)*np.array(list(self.Q[s_pr].values())))
            
            self.Q[s][a] = (self.Q[s][a] + self.config['alpha']
                                          *(r_s_pr_a_s + self.config['gamma']*expectation - self.Q[s][a]))    #perform the expected SARSA update


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
def compute_boards(config,n_tries):
    '''A function to pre-compute all possible board positions
        Takes: 
            config -- the agent config dictionary
            n_tries -- int, the number of simulations to do
    '''
    board = Battleship(config)    #set up the environment
    boards = {}    #a dictionary to hold possible boards
    for _ in tqdm(range(n_tries)):    #loop over simulations
        board.make_board()    #make a new board
        k = ''    #initialize the key
        for xy in list(board.true_state.flatten()):    #loop over true board values
            k = k + str(xy)    #add to the key
        if k not in boards:    #if this board hasn't been seen yet...
            boards[k] = copy.deepcopy(board.true_state)    #save it
        board.reset()    #reset the environment
    return boards


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#set up the config file with precomputed boards
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
config = {'n':6,'ships':[2,3,4],'boards':None}    #set up the environmental config
boards = compute_boards(config,1000000)    #pre-compute boards; should be 49504 of them
config['boards'] = boards    #save these precomputed boards
print('Found',len(list(boards.keys())),'states.')


In [ ]:
# board_out = []
# for j in np.random.choice(range(len(list(boards.keys()))),100):
#     board_out.append(boards[list(boards.keys())[j]])

# import pickle
# with open('all_boards.pkl', 'wb') as f:
#     pickle.dump({'boards':boards},f)
   

In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#traing the agent with Q-learning or expected SARSA
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
board = Battleship(config)    #set up the board
agent_config = {'n':6,'top_k':10,'epsilon':0.2,'gamma':0.9,'alpha':0.1}    #set up the player config
player = agent(agent_config)    #set up the player

n_turns = []
n_episodes = 2    #set the number of episodes
for episode in tqdm(range(n_episodes)):    #loop over episodes
    board.make_board()    #set up the board
    update = {'state':board.state,'reward':1,'done':False}    #record the board state information
    s = round(np.log(np.sum(update['state'])+1))    #initialize the current state in agent terms
    count = 0
    while not update['done']:    #while we continue...
        print(s)
        a = player.pi(s,agent_config['epsilon'])    #choose an action -- a threshold for sampling coordinates
        coordinates = player.shoot_where(state = board.state,a = a)    #turn the threshold into actual coordinates
        coordinates = {'row':coordinates[0]
                ,'col':coordinates[1]}    #put into format for the step function
        update = board.step(coordinates)    #call the step function to evolve the state
        r_s_pr_a_s = update['reward']    #record the new reward
        s_pr = round(np.log(np.sum(update['state'])+1))    #record the new state in agent terms
        player.update_Q_learning(s, a, r_s_pr_a_s, s_pr)    #apply the Q-learning update rule
        #player.update_Expected_SARSA(s, a, r_s_pr_a_s, s_pr)
        s = s_pr    #update the current state
        print(board.state)
        count += 1
    board.reset()    #reset for a new game
    n_turns.append(count)

In [ ]:
np.mean(n_turns)

In [ ]:
2**9